In [5]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
import time
import pandas as pd

### Configurações Iniciais

In [6]:
# Lista de produtos para pesquisa
PRODUTOS_ELETRONICOS = [
    "Caixa de Som JBL Flip 6",
    "Smart TV Samsung 50 polegadas Crystal UHD",
    "iPhone 15 Apple 128GB",
    "Smartphone Samsung Galaxy S23 Ultra",
    "Tablet Apple iPad Air M2",
    "Notebook Gamer Acer Nitro 5",
    "Monitor Gamer LG Ultragear 27",
    "Mouse Sem Fio Logitech G305",
    "Teclado Mecânico Razer BlackWidow",
    "Headset Gamer HyperX Cloud II",
    "Fone Bluetooth Sony WH-1000XM5",
    "Caixa de som JBL Boombox 3",
    "Apple Watch Series 9",
    "Console PlayStation 5 Slim",
    "Console Xbox Series X",
    "Placa de Vídeo RTX 4060 NVIDIA",
    "Memória RAM Kingston Fury 8GB DDR4",
    "SSD Kingston NV2 1TB NVMe",
    "Processador Intel Core i5-13400F",
    "Webcam Logitech C920 Full HD",
    "Roteador Wi-Fi 6 TP-Link Archer",
    "Impressora Epson EcoTank L3250",
    "Carregador Portátil Baseus 20000mAh",
    "Cabo HDMI 2.1 8K Baseus",
    "Microfone Condensador HyperX QuadCast",
    "Suporte para Monitor articulado F80N"
]

# Armazenamento dos resultados
lista_produtos = []

# Configuração do Navegador
navegador = webdriver.Chrome()
navegador.maximize_window()

# URL de referência
url_kabum = "https://www.kabum.com.br"
url_amazon = "https://www.amazon.com.br"

### Web Scraping - Kabum

In [7]:
# Acesso inicial ao site
navegador.get(url_kabum)
time.sleep(5)

# Percorre cada item da lista de eletrônicos
for item_pesquisa in PRODUTOS_ELETRONICOS:
    try:
        print(f"\n>>> Pesquisando por: {item_pesquisa}...")
        
        # 1. Localiza o campo de busca
        busca = navegador.find_element("xpath", "//*[@id='inputBusca']")
        
        # 2. Limpa o campo e digita o novo item
        busca.send_keys(Keys.CONTROL + "a")
        busca.send_keys(Keys.DELETE)
        busca.send_keys(item_pesquisa)
        busca.send_keys(Keys.ENTER)

        # 3. Aguarda o carregamento dos resultados
        time.sleep(7)

        # 4. Captura os elementos e limita aos 3 primeiros
        nomes_elementos = navegador.find_elements("class name", "h-40")[:3]
        
        precos_elementos = navegador.find_elements(
            "xpath", 
            "//div[contains(@class, 'flex gap-4 items-center')]/span[2]"
        )[:3]

        links_elementos = navegador.find_elements(
            "css selector",
            "a.flex.flex-col.relative.gap-4"
        )[:3]

        # 5. Loop para organizar os produtos encontrados
        for i in range(len(nomes_elementos)):
            try:
                nome = nomes_elementos[i].text
                valor = precos_elementos[i].text if i < len(precos_elementos) else "Sem preço"
                link = links_elementos[i].get_attribute("href") if i < len(links_elementos) else "Sem link"
                
                # Adiciona ao dicionário (cada item vira uma linha no DataFrame)
                lista_produtos.append({
                    "Pesquisa": item_pesquisa,
                    "Produto": nome,
                    "Preço": valor,
                    "Link": link,
                    "Loja": "KaBuM"
                })
                
                print(f"Encontrado: {nome[:50]}... | Valor: {valor}")
                
            except Exception as e:
                print(f"Erro ao processar item {i}: {e}")

    except Exception as e:
        print(f"Erro na busca do termo '{item_pesquisa}': {e}")

print("\n>>> Pesquisa concluída. Indo para próxima etapa...")


>>> Pesquisando por: Caixa de Som JBL Flip 6...
Encontrado: Caixa de Som Bluetooth Portátil JBL Flip 7, 35W RM... | Valor: 749,00
Encontrado: Caixa De Som Portátil JBL Flip 6, Bluetooth, 20W R... | Valor: 889,00
Encontrado: Caixa de Som Bluetooth Portátil Charge 6 JBL - Pre... | Valor: 996,98

>>> Pesquisando por: Smart TV Samsung 50 polegadas Crystal UHD...
Encontrado: Samsung Smart TV 50" Crystal UHD 4K U8600F 2025, X... | Valor: 2.219,90
Encontrado: Smart TV 50" Samsung UHD 4K Crystal UHD U8600F UN5... | Valor: 2.688,00
Encontrado: Samsung Smart Tv 58” Crystal Uhd 4k U8500f 2025, X... | Valor: 2.789,90

>>> Pesquisando por: iPhone 15 Apple 128GB...
Encontrado: Iphone 15 Apple, 128GB, Quadriband, 6,1 Polegadas,... | Valor: 4.499,10
Encontrado: Apple Iphone 15 128GB Azul... | Valor: 4.299,00
Encontrado: Apple Iphone 15 128GB Rosa... | Valor: 4.299,00

>>> Pesquisando por: Smartphone Samsung Galaxy S23 Ultra...
Encontrado: Usado - Smartphone Samsung Galaxy S23 Ultra, 5G, 5... | Valor:

In [8]:
navegador.get(url_amazon)

time.sleep(5)

for item_pesquisa in PRODUTOS_ELETRONICOS:
    try:
        print(f"\n>>> Pesquisando por: {item_pesquisa}...")
        
        busca = navegador.find_element("xpath", "//*[@id='twotabsearchtextbox']")
        
        busca.send_keys(Keys.CONTROL + "a")
        busca.send_keys(Keys.DELETE)
        busca.send_keys(item_pesquisa)
        busca.send_keys(Keys.ENTER)

        time.sleep(7)

        nomes_elementos = navegador.find_elements("css selector", "h2.a-size-base-plus span")[:3]

        precos_elementos = navegador.find_elements("css selector", "span.a-price")[:3]

        links_elementos = navegador.find_elements("css selector", "a.a-link-normal.s-line-clamp-4")[:3]

        for i in range(len(nomes_elementos)):
            try:
                nome = nomes_elementos[i].text
                
                try:
                    inteiro = precos_elementos[i].find_element("css selector", ".a-price-whole").text
                    centavos = precos_elementos[i].find_element("css selector", ".a-price-fraction").text
                    inteiro = inteiro.replace(",", "").replace(".", "")
                    valor = f"R$ {inteiro},{centavos}"
                except:
                    valor = "Sem preço"

                link = links_elementos[i].get_attribute("href")

                lista_produtos.append({
                    "Pesquisa": item_pesquisa,
                    "Produto": nome,
                    "Preço": valor,
                    "Link": link,
                    "Loja": "Amazon"
                })
                
                print(f"Encontrado: {nome[:50]}... | Valor: {valor}")
                
            except Exception as e:
                print(f"Erro ao processar item {i}: {e}")

    except Exception as e:
        print(f"Erro na busca do termo '{item_pesquisa}': {e}")

if lista_produtos:
    df = pd.DataFrame(lista_produtos)

    print("\n" + "="*60)
    print("PRÉVIA DOS DADOS COLETADOS:")
    print(df.head())

    df.to_csv("resultados.csv", index=False, encoding='utf-8-sig', sep=';')

    print("\n" + "="*60)
    print(f"ARQUIVO 'resultados_amazon.csv' GERADO COM SUCESSO!")
    print("="*60)
else:
    print("Nenhum produto foi capturado.")

time.sleep(2)
navegador.quit()


>>> Pesquisando por: Caixa de Som JBL Flip 6...
Encontrado: JBL, Caixa de Som, FLIP 7 - Preta... | Valor: R$ 669,00
Encontrado: JBL, Caixa de Som, PartyBox Encore Essential 2, Sh... | Valor: Sem preço
Encontrado: JBL, Caixa de Som, Go 4, Bluetooth, Portátil, Aura... | Valor: Sem preço

>>> Pesquisando por: Smart TV Samsung 50 polegadas Crystal UHD...
Encontrado: Samsung Smart TV 43" Crystal UHD 4K U8600F 2025... | Valor: R$ 1849,00
Encontrado: TV LG 32" LED HD Smart Pro 32RL601CBSA... | Valor: Sem preço
Encontrado: Samsung Smart TV 43" FHD F6000F 2025... | Valor: Sem preço

>>> Pesquisando por: iPhone 15 Apple 128GB...
Encontrado: Apple iPhone 15 (128 GB) — Preto... | Valor: R$ 59,00
Encontrado: Apple iPhone 16 (128 GB) – Branco (Recondicionado)... | Valor: R$ 69,00
Encontrado: Cabo USB-C para USB-C 100W PD 5A 9 Carregamento Ul... | Valor: R$ 59,00

>>> Pesquisando por: Smartphone Samsung Galaxy S23 Ultra...
Encontrado: Smartphone Samsung Galaxy S24 Ultra, Galaxy AI, Se... | Valor: R$